# EXP-029: 변이유형 구성비·log burden 피처

Issue #29 실험 노트북입니다. EXP-005의 유전자×변이유형 피처와 XGBoost 설정은 유지하고, 사전에 정의한 log burden 및 구성비 피처 묶음의 OOF Macro F1 변화를 확인합니다.

- 상관분석·Mutual Information·피처 선택은 수행하지 않습니다.
- 원본 CSV와 공용 split은 수정하지 않습니다.
- 실행 전에는 결과나 점수를 가정하지 않습니다.

## 1. 로컬 환경 준비

현재 저장소의 Issue #29 브랜치에서 실행합니다. 저장소 루트 또는 `notebooks/` 디렉터리에서 Jupyter로 열 수 있으며, 원본 CSV는 로컬 `data/raw/`에 있어야 합니다.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from scipy import sparse

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "저장소 루트에서 실행하세요."
ROOT

## 2. 로컬 전용 원본 데이터 확인

주최측에서 받은 세 CSV를 다음 경로에 놓습니다.

- `data/raw/train.csv`
- `data/raw/test.csv`
- `data/raw/sample_submission.csv`

In [ ]:
raw_paths = {
    "train": ROOT / "data/raw/train.csv",
    "test": ROOT / "data/raw/test.csv",
    "sample_submission": ROOT / "data/raw/sample_submission.csv",
}
missing = [str(path) for path in raw_paths.values() if not path.is_file()]
assert not missing, f"원본 파일을 배치하세요: {missing}"
pd.Series({name: path.stat().st_size for name, path in raw_paths.items()}, name="bytes")

## 3. EXP-029 피처 생성

EXP-005 피처에 log burden 3개와 구성비 7개를 추가합니다. `SUBCLASS`는 피처 계산에 사용하지 않습니다.

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from open_cancer.mutation_features import (
    GLOBAL_FEATURES,
    ROBUST_GLOBAL_FEATURES,
    build_mutation_features,
)

feature_dir = ROOT / "data/processed/exp029_log_burden_ratios"
feature_report = build_mutation_features(
    raw_paths["train"],
    raw_paths["test"],
    feature_dir,
    include_robust_aggregates=True,
)
feature_report["train"], feature_report["test"], feature_report["feature_count"]

## 4. 사전 안전성 검사

상관분석은 하지 않고 계산 범위·0분모·결측·train/test 피처 순서만 확인합니다.

In [ ]:
feature_names = json.loads((feature_dir / "feature_names.json").read_text())
train_x = sparse.load_npz(feature_dir / "train_features.npz")
test_x = sparse.load_npz(feature_dir / "test_features.npz")
aggregate_names = [*GLOBAL_FEATURES, *ROBUST_GLOBAL_FEATURES]
aggregate_indices = [feature_names.index(name) for name in aggregate_names]
train_agg = pd.DataFrame(train_x[:, aggregate_indices].toarray(), columns=aggregate_names)
test_agg = pd.DataFrame(test_x[:, aggregate_indices].toarray(), columns=aggregate_names)

ratio_columns = [name for name in ROBUST_GLOBAL_FEATURES if name.endswith("_ratio")]
assert np.isfinite(train_agg.to_numpy()).all()
assert np.isfinite(test_agg.to_numpy()).all()
assert train_agg[ratio_columns].ge(0).all().all()
assert train_agg[ratio_columns].le(1).all().all()
assert test_agg[ratio_columns].ge(0).all().all()
assert test_agg[ratio_columns].le(1).all().all()
assert train_x.shape[1] == test_x.shape[1] == len(feature_names)

pd.concat(
    {"train": train_agg.describe().T, "test": test_agg.describe().T},
    axis=1,
).loc[list(ROBUST_GLOBAL_FEATURES)]

## 5. 공용 5-fold 내부 검증 실행

아래 셀은 모델 5개를 학습하고 OOF·test 확률·제출 후보·metrics를 생성합니다. 실행 중에는 노트북을 종료하지 마세요.

In [ ]:
command = [sys.executable, str(ROOT / "scripts/run_exp029_xgb_log_burden_ratios.py")]
completed = subprocess.run(command, cwd=ROOT, check=True, text=True)
completed.returncode

## 6. EXP-005와 실제 결과 비교

전체 OOF Macro F1을 1순위로 보고 fold별 안정성과 클래스별 변화를 함께 확인합니다.

In [ ]:
exp005 = json.loads((ROOT / "reports/exp005_xgb_mutation_features/metrics.json").read_text())
exp029 = json.loads((ROOT / "reports/exp029_xgb_log_burden_ratios/metrics.json").read_text())

summary = pd.DataFrame(
    {
        "EXP-005": {
            "OOF Macro F1": exp005["oof"]["macro_f1"],
            "Fold mean": exp005["oof"]["fold_mean"],
            "Fold std": exp005["oof"]["fold_std"],
            "Log Loss": exp005["oof"]["log_loss"],
        },
        "EXP-029": {
            "OOF Macro F1": exp029["oof"]["macro_f1"],
            "Fold mean": exp029["oof"]["fold_mean"],
            "Fold std": exp029["oof"]["fold_std"],
            "Log Loss": exp029["oof"]["log_loss"],
        },
    }
)
summary["difference"] = summary["EXP-029"] - summary["EXP-005"]
display(summary)

class_delta = pd.Series(exp029["oof"]["per_class_f1"]) - pd.Series(exp005["oof"]["per_class_f1"])
display(class_delta.sort_values().rename("EXP-029 - EXP-005 class F1"))